In [1]:
# Kaggle setup: clone repo, install libraries, find the data,
# copy results.csv and the trained Faster R-CNN weights from the 02 output.
import os, glob, shutil, subprocess

REPO = "polluvl/hw2-video-detection"

if os.path.isdir("/kaggle/working"):
    # 1. clone the repo once per session
    if not os.path.isdir("/kaggle/working/repo"):
        r = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/kaggle/working/repo"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed (is Internet on?):\n" + r.stderr)
    # 2. install the pinned libraries
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
    os.chdir("/kaggle/working/repo/notebooks")

    # 3. find where Kaggle mounted the competition data
    hits = glob.glob("/kaggle/input/**/test_images.csv", recursive=True)
    if not hits:
        raise RuntimeError("Competition data not found: add it in the right panel under Input")
    os.environ["DATA_ROOT"] = os.path.dirname(hits[0])
    print("data root:", os.environ["DATA_ROOT"])

    # 4. results.csv and weights from the 02_faster_rcnn output (attached as input)
    found = glob.glob("/kaggle/input/**/preds_val_frcnn_base.npz", recursive=True)
    if not found:
        raise RuntimeError("02 output not found: Add Input -> Your Work -> 02_faster_rcnn")
    src = os.path.dirname(found[0])
    shutil.copy(f"{src}/results.csv", "/kaggle/working/results.csv")
    for f in glob.glob(f"{src}/frcnn_*.pt"):
        shutil.copy(f, "/kaggle/working/")
        print("copied", os.path.basename(f))

data root: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
copied frcnn_base.pt
copied frcnn_giou.pt
copied frcnn_roi.pt


In [2]:
# Kaggle submission from the best Faster R-CNN run of 02 (chosen by our val mAP)
import ast
import sys
from pathlib import Path

import pandas as pd
import torch

sys.path.insert(0, str(Path("../src").resolve()))
from common import find_data_root, list_test_frames, save_preds, work_dir, write_submission
from tv_detection import build_faster_rcnn, predict

assert torch.cuda.is_available(), "No GPU! Session options -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
DATA = find_data_root()
OUT = work_dir()
DEVICE = torch.device("cuda")

# best Faster R-CNN run by val mAP, rebuilt with the same settings it was trained with
results = pd.read_csv(OUT / "results.csv")
frcnn = results[results.experiment.str.startswith("frcnn")].sort_values("mAP")
best_name = frcnn.experiment.iloc[-1]
changes = ast.literal_eval(frcnn.config.iloc[-1])
print("best run:", best_name, "| val mAP", frcnn.mAP.iloc[-1], "| changes", changes)
model = build_faster_rcnn({"min_size": 800, "max_size": 1333, **changes}, pretrained=False)
model.load_state_dict(torch.load(OUT / f"{best_name}.pt", map_location=DEVICE))
model.to(DEVICE)

# predict on all test frames (test has no labels: empty label/ignore paths)
test_frames = list_test_frames(DATA)
test_frames["label_path"] = ""
test_frames["ignore_path"] = ""
test_preds, ms = predict(model, test_frames, DEVICE)
save_preds(test_preds, OUT / f"preds_test_{best_name}.npz")
sub = write_submission(test_frames, test_preds, OUT / "submission.csv")
print(f"submission.csv: {len(sub)} rows from {best_name}, {ms:.0f} ms/img")

GPU: Tesla T4
best run: frcnn_base | val mAP 0.2384 | changes {}
submission.csv: 3811 rows from frcnn_base, 251 ms/img
